In [11]:
"""
================================================================================
  CLAHE Medical Imaging — EDA Pipeline
  Thiết kế cho cấu trúc dataset:

      root/
      ├── images/
      │   ├── sample_001/
      │   │   ├── view1.jpg   ← góc chụp 1
      │   │   └── view2.jpg   ← góc chụp 2
      │   └── sample_002/ ...
      └── annotation.json
              [{id, report, image_path: [path_v1, path_v2], split}, ...]

Cài đặt:
    pip install opencv-python numpy matplotlib seaborn pandas scikit-image tqdm
================================================================================
"""

import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim
import warnings
warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────
#  CẤU HÌNH — chỉnh theo dataset của bạn
# ─────────────────────────────────────────────
CONFIG = {
    "annotation_path"  : r"G:\My Drive\NCKH\A3Net-main\data\iu_xray\annotation.json",  # Đường dẫn file JSON
    "image_root"       : r"G:\My Drive\NCKH\A3Net-main\data\iu_xray\images",                 # Root để resolve đường dẫn trong JSON
    "output_dir"       : "eda_output",
    "clahe_clip"       : 2.0,
    "clahe_tile"       : (8, 8),
    "sample_per_split" : 4,                   # Số mẫu hiển thị cho mỗi split
    "dpi"              : 150,
    "splits"           : ["train", "val", "test"],
    "split_colors"     : {"train": "#4C9BE8", "val": "#F4A261", "test": "#57CC99"},
}


# ─────────────────────────────────────────────
#  TIỆN ÍCH
# ─────────────────────────────────────────────

def setup_dirs(output_dir: str) -> Path:
    out = Path(output_dir)
    for sub in ["1_before_after", "2_histograms", "3_statistics", "4_split_analysis"]:
        (out / sub).mkdir(parents=True, exist_ok=True)
    return out


def load_annotations(ann_path: str) -> pd.DataFrame:
    """Đọc annotation.json → DataFrame chuẩn hoá."""
    with open(ann_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    # data = {"train": [...], "val": [...], "test": [...]}
    records = []
    for split, items in data.items():
        for item in items:
            paths = item["image_path"]
            records.append({
                "id"     : item["id"],
                "report" : item.get("report", ""),
                "split"  : split,               # ← lấy từ key thay vì field
                "path_v1": paths[0] if len(paths) > 0 else None,
                "path_v2": paths[1] if len(paths) > 1 else None,
            })

    df = pd.DataFrame(records)
    print(f"✅ Đã load {len(df)} mẫu")
    for sp in ["train", "val", "test"]:
        n = (df["split"] == sp).sum()
        if n > 0:
            print(f"   • {sp:6s}: {n} mẫu")
    return df


def read_img(path: str, root: str = "."):
    """Đọc ảnh — hỗ trợ cả absolute & relative path."""
    p = Path(path)
    if not p.is_absolute():
        p = Path(root) / p
    img = cv2.imread(str(p))
    if img is None:
        print(f"   ⚠️  Không đọc được: {p}")
    return img


def apply_clahe(img_bgr, clip: float, tile: tuple):
    """Áp dụng CLAHE trên kênh L (LAB), bảo toàn màu sắc."""
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=tile)
    if len(img_bgr.shape) == 2:           # grayscale
        return clahe.apply(img_bgr)
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    return cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]), cv2.COLOR_LAB2BGR)


def to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if len(img.shape) == 3 else img


def to_gray(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img


def calc_entropy(gray) -> float:
    h, _ = np.histogram(gray.ravel(), 256, [0, 256])
    h = h / h.sum()
    h = h[h > 0]
    return float(-np.sum(h * np.log2(h)))


# ─────────────────────────────────────────────
#  1. SO SÁNH TRƯỚC / SAU CLAHE  (cả 2 view)
# ─────────────────────────────────────────────

def plot_before_after(df: pd.DataFrame, out: Path, cfg: dict):
    """
    Layout mỗi hàng (1 mẫu):
        [View1 Gốc] [View1 CLAHE] | [View2 Gốc] [View2 CLAHE] | [Diff Map V1]
    Xuất file riêng cho mỗi split.
    """
    print("\n📸 [1/4] So sánh Trước/Sau CLAHE...")
    root = cfg["image_root"]

    for split in cfg["splits"]:
        subset = df[df["split"] == split]
        if subset.empty:
            continue
        sample = subset.sample(min(cfg["sample_per_split"], len(subset)), random_state=42)
        n = len(sample)

        fig, axes = plt.subplots(n, 5, figsize=(22, n * 4.5))
        if n == 1:
            axes = [axes]

        fig.suptitle(
            f"So sánh Trước / Sau CLAHE  —  Split: {split.upper()}  "
            f"(clip={cfg['clahe_clip']}, tile={cfg['clahe_tile']})",
            fontsize=14, fontweight="bold", y=1.01
        )
        col_titles = ["View 1 — Gốc", "View 1 — CLAHE",
                      "View 2 — Gốc", "View 2 — CLAHE",
                      "Diff Map (V1)"]

        for i, (_, row) in enumerate(sample.iterrows()):
            v1 = read_img(row["path_v1"], root)
            v2 = read_img(row["path_v2"], root)
            if v1 is None or v2 is None:
                continue

            v1c = apply_clahe(v1, cfg["clahe_clip"], cfg["clahe_tile"])
            v2c = apply_clahe(v2, cfg["clahe_clip"], cfg["clahe_tile"])

            g1, g1c = to_gray(v1), to_gray(v1c)
            score   = ssim(g1, g1c, data_range=255)
            diff    = cv2.absdiff(g1, g1c)

            for j, img in enumerate([v1, v1c, v2, v2c]):
                axes[i][j].imshow(to_rgb(img))
                axes[i][j].axis("off")
                if i == 0:
                    axes[i][j].set_title(col_titles[j], fontsize=9, fontweight="bold")

            imd = axes[i][4].imshow(diff, cmap="hot")
            axes[i][4].axis("off")
            if i == 0:
                axes[i][4].set_title(col_titles[4], fontsize=9, fontweight="bold")
            plt.colorbar(imd, ax=axes[i][4], fraction=0.046, pad=0.04)

            axes[i][0].set_ylabel(
                f"ID: {row['id']}\nSSIM V1: {score:.4f}",
                fontsize=8, rotation=0, labelpad=90, va="center"
            )

        plt.tight_layout()
        save_path = out / "1_before_after" / f"before_after_{split}.png"
        plt.savefig(save_path, dpi=cfg["dpi"], bbox_inches="tight")
        plt.close()
        print(f"   ✔ {split:5s} → {save_path}")


# ─────────────────────────────────────────────
#  2. HISTOGRAM  (PDF + CDF, theo split & view)
# ─────────────────────────────────────────────

def plot_histograms(df: pd.DataFrame, out: Path, cfg: dict):
    print("\n📊 [2/4] Phân tích Histogram...")
    root     = cfg["image_root"]
    s_colors = cfg["split_colors"]

    # ── 2a. PDF Overlay tất cả split  (2 view × 2 phase = 4 subplot)
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle("PDF Pixel Intensity — Overlay tất cả Split",
                 fontsize=13, fontweight="bold")

    for split in cfg["splits"]:
        subset = df[df["split"] == split].sample(
            min(30, len(df[df["split"] == split])), random_state=42)
        h = {f"v{v}_{ph}": np.zeros(256)
             for v in [1, 2] for ph in ["orig", "clahe"]}

        for _, row in subset.iterrows():
            for v, pk in [(1, "path_v1"), (2, "path_v2")]:
                img = read_img(row[pk], root)
                if img is None:
                    continue
                img_c  = apply_clahe(img, cfg["clahe_clip"], cfg["clahe_tile"])
                ho, _  = np.histogram(to_gray(img).ravel(),   256, [0, 256])
                hc, _  = np.histogram(to_gray(img_c).ravel(), 256, [0, 256])
                h[f"v{v}_orig"]  += ho
                h[f"v{v}_clahe"] += hc

        color = s_colors.get(split, "gray")
        for v_i, (ax_orig, ax_clahe) in enumerate(zip(axes[:, 0], axes[:, 1]), start=1):
            if v_i == 1:
                ax_o, ax_c = axes[0][0], axes[0][1]
            else:
                ax_o, ax_c = axes[1][0], axes[1][1]

            for ax, ph in [(ax_o, "orig"), (ax_c, "clahe")]:
                key  = f"v{v_i}_{ph}"
                norm = h[key] / h[key].sum() if h[key].sum() > 0 else h[key]
                ax.plot(norm, color=color, alpha=0.8, linewidth=1.5, label=split)

    subtitles = [["View 1 — Gốc", "View 1 — CLAHE"],
                 ["View 2 — Gốc", "View 2 — CLAHE"]]
    for r in range(2):
        for c in range(2):
            axes[r][c].set_title(subtitles[r][c], fontweight="bold")
            axes[r][c].set_xlabel("Pixel Intensity")
            axes[r][c].set_ylabel("Frequency (norm)")
            axes[r][c].set_xlim(0, 255)
            axes[r][c].grid(alpha=0.3)
            handles, labels = axes[r][c].get_legend_handles_labels()
            axes[r][c].legend(dict(zip(labels, handles)).values(),
                              dict(zip(labels, handles)).keys(), title="Split")

    plt.tight_layout()
    plt.savefig(out / "2_histograms" / "pdf_overlay_all_splits.png",
                dpi=cfg["dpi"], bbox_inches="tight")
    plt.close()

    # ── 2b. PDF + CDF riêng từng split
    for split in cfg["splits"]:
        subset = df[df["split"] == split].sample(
            min(30, len(df[df["split"] == split])), random_state=42)
        if subset.empty:
            continue

        fig, axes = plt.subplots(2, 2, figsize=(14, 9))
        fig.suptitle(f"PDF & CDF — Split: {split.upper()}", fontsize=13, fontweight="bold")

        for v_idx, pk in enumerate(["path_v1", "path_v2"], start=1):
            h_orig = np.zeros(256)
            h_clahe = np.zeros(256)

            for _, row in subset.iterrows():
                img = read_img(row[pk], root)
                if img is None:
                    continue
                img_c  = apply_clahe(img, cfg["clahe_clip"], cfg["clahe_tile"])
                ho, _  = np.histogram(to_gray(img).ravel(),   256, [0, 256])
                hc, _  = np.histogram(to_gray(img_c).ravel(), 256, [0, 256])
                h_orig += ho; h_clahe += hc

            # PDF
            ax_pdf = axes[v_idx - 1][0]
            ax_cdf = axes[v_idx - 1][1]
            ax_pdf.plot(h_orig  / h_orig.sum(),  color="#5B8DB8", linewidth=2, label="Gốc")
            ax_pdf.plot(h_clahe / h_clahe.sum(), color="#E07B54", linewidth=2, label="CLAHE")
            ax_pdf.set_title(f"View {v_idx} — PDF", fontweight="bold")
            ax_pdf.set_xlabel("Intensity"); ax_pdf.set_ylabel("Frequency (norm)")
            ax_pdf.set_xlim(0, 255); ax_pdf.legend(); ax_pdf.grid(alpha=0.3)

            # CDF
            cdf_o = np.cumsum(h_orig)  / h_orig.sum()
            cdf_c = np.cumsum(h_clahe) / h_clahe.sum()
            ax_cdf.plot(cdf_o, color="#5B8DB8", linewidth=2, label="Gốc")
            ax_cdf.plot(cdf_c, color="#E07B54", linewidth=2, label="CLAHE")
            ax_cdf.fill_between(range(256), cdf_o, cdf_c,
                                alpha=0.15, color="green", label="Δ area")
            ax_cdf.set_title(f"View {v_idx} — CDF", fontweight="bold")
            ax_cdf.set_xlabel("Intensity"); ax_cdf.set_ylabel("CDF")
            ax_cdf.set_xlim(0, 255); ax_cdf.set_ylim(0, 1)
            ax_cdf.legend(); ax_cdf.grid(alpha=0.3)

        plt.tight_layout()
        plt.savefig(out / "2_histograms" / f"pdf_cdf_{split}.png",
                    dpi=cfg["dpi"], bbox_inches="tight")
        plt.close()
        print(f"   ✔ {split:5s} → PDF + CDF saved")


# ─────────────────────────────────────────────
#  3. THỐNG KÊ PIXEL
# ─────────────────────────────────────────────

def compute_stats(df: pd.DataFrame, out: Path, cfg: dict) -> pd.DataFrame:
    """
    Tính mean, std, min, max, entropy cho:
        - View 1 & View 2
        - Phase: gốc (orig) & CLAHE
        - Delta = CLAHE − gốc
    """
    print("\n📐 [3/4] Tính thống kê pixel...")
    root = cfg["image_root"]
    records = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="   Stats"):
        rec = {"id": row["id"], "split": row["split"]}

        for v_idx, pk in enumerate(["path_v1", "path_v2"], start=1):
            img = read_img(row[pk], root)
            if img is None:
                for ph in ["orig", "clahe"]:
                    for m in ["mean", "std", "min", "max", "entropy"]:
                        rec[f"v{v_idx}_{ph}_{m}"] = np.nan
                continue

            img_c = apply_clahe(img, cfg["clahe_clip"], cfg["clahe_tile"])
            g, gc = to_gray(img), to_gray(img_c)

            for ph, arr in [("orig", g), ("clahe", gc)]:
                rec[f"v{v_idx}_{ph}_mean"]    = float(arr.mean())
                rec[f"v{v_idx}_{ph}_std"]     = float(arr.std())
                rec[f"v{v_idx}_{ph}_min"]     = int(arr.min())
                rec[f"v{v_idx}_{ph}_max"]     = int(arr.max())
                rec[f"v{v_idx}_{ph}_entropy"] = calc_entropy(arr)

            for m in ["mean", "std", "entropy"]:
                rec[f"v{v_idx}_delta_{m}"] = (
                    rec[f"v{v_idx}_clahe_{m}"] - rec[f"v{v_idx}_orig_{m}"]
                )
        records.append(rec)

    stats_df = pd.DataFrame(records)
    csv_path  = out / "3_statistics" / "pixel_stats.csv"
    stats_df.to_csv(csv_path, index=False)
    print(f"   ✔ CSV → {csv_path}")

    # ── Boxplot: mean / std / entropy
    for metric in ["mean", "std", "entropy"]:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
        fig.suptitle(
            f"Phân phối {metric.capitalize()} — Gốc vs CLAHE theo Split",
            fontsize=12, fontweight="bold"
        )
        for ax, v in zip(axes, [1, 2]):
            rows = []
            for _, r in stats_df.iterrows():
                for ph in ["orig", "clahe"]:
                    val = r.get(f"v{v}_{ph}_{metric}", np.nan)
                    if not np.isnan(val):
                        rows.append({"Split": r["split"], "Phase": ph.upper(), metric: val})
            if not rows:
                continue
            sns.boxplot(
                data=pd.DataFrame(rows), x="Split", y=metric, hue="Phase",
                order=cfg["splits"],
                palette={"ORIG": "#5B8DB8", "CLAHE": "#E07B54"}, ax=ax
            )
            ax.set_title(f"View {v}", fontweight="bold")
            ax.set_xlabel("Split"); ax.set_ylabel(metric.capitalize())
            ax.grid(axis="y", alpha=0.3)

        plt.tight_layout()
        plt.savefig(out / "3_statistics" / f"boxplot_{metric}.png",
                    dpi=cfg["dpi"], bbox_inches="tight")
        plt.close()

    # ── Delta KDE  (2 view × 3 metric = 2×3 grid)
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle("Phân bố Δ = CLAHE − Gốc (theo Split & View)",
                 fontsize=13, fontweight="bold")

    for v_idx, row_axes in enumerate(axes, start=1):
        for ax, metric in zip(row_axes, ["mean", "std", "entropy"]):
            for split in cfg["splits"]:
                col  = f"v{v_idx}_delta_{metric}"
                data = stats_df[stats_df["split"] == split][col].dropna() \
                       if col in stats_df.columns else pd.Series()
                if not data.empty:
                    sns.kdeplot(data, ax=ax, label=split,
                                color=cfg["split_colors"].get(split, "gray"),
                                fill=True, alpha=0.25)
            ax.axvline(0, color="red", linestyle="--", linewidth=1.5)
            ax.set_title(f"View {v_idx} — Δ {metric.capitalize()}", fontweight="bold")
            ax.set_xlabel(f"Δ {metric}"); ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(out / "3_statistics" / "delta_kde.png",
                dpi=cfg["dpi"], bbox_inches="tight")
    plt.close()
    print("   ✔ Boxplots + Delta KDE saved")

    return stats_df


# ─────────────────────────────────────────────
#  4. PHÂN TÍCH THEO SPLIT
# ─────────────────────────────────────────────

def plot_split_analysis(df: pd.DataFrame, stats_df: pd.DataFrame,
                        out: Path, cfg: dict):
    print("\n📈 [4/4] Phân tích theo Split...")
    s_colors = cfg["split_colors"]
    splits   = cfg["splits"]

    # ── 4a. Phân bố số mẫu
    split_counts = df["split"].value_counts().reindex(splits).fillna(0)
    colors = [s_colors.get(s, "gray") for s in split_counts.index]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("Phân bố Số Mẫu theo Split", fontsize=13, fontweight="bold")

    axes[0].bar(split_counts.index, split_counts.values,
                color=colors, edgecolor="white", linewidth=1.5)
    for i, (sp, cnt) in enumerate(split_counts.items()):
        axes[0].text(i, cnt + 0.3, str(int(cnt)),
                     ha="center", va="bottom", fontweight="bold")
    axes[0].set_xlabel("Split"); axes[0].set_ylabel("Số mẫu")
    axes[0].grid(axis="y", alpha=0.3)

    axes[1].pie(split_counts.values, labels=split_counts.index, colors=colors,
                autopct="%1.1f%%", startangle=90,
                wedgeprops=dict(edgecolor="white", linewidth=2))
    axes[1].set_title("Tỉ lệ phân bố (%)", fontweight="bold")

    plt.tight_layout()
    plt.savefig(out / "4_split_analysis" / "split_distribution.png",
                dpi=cfg["dpi"], bbox_inches="tight")
    plt.close()

    # ── 4b. Scatter: View1 Mean (CLAHE) vs View2 Mean (CLAHE)
    # Kiểm tra mức độ tương đồng giữa 2 góc chụp sau CLAHE
    fig, ax = plt.subplots(figsize=(8, 7))
    for split in splits:
        sub = stats_df[stats_df["split"] == split]
        ax.scatter(sub["v1_clahe_mean"], sub["v2_clahe_mean"],
                   label=split, color=s_colors.get(split, "gray"),
                   alpha=0.6, s=50, edgecolors="none")

    ax.set_title("Tương quan Mean Intensity (sau CLAHE)\nView 1 vs View 2",
                 fontweight="bold")
    ax.set_xlabel("View 1 — CLAHE Mean")
    ax.set_ylabel("View 2 — CLAHE Mean")
    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]),
            max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, "r--", alpha=0.5, linewidth=1.5, label="y = x")
    ax.legend(title="Split"); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(out / "4_split_analysis" / "scatter_view1_vs_view2.png",
                dpi=cfg["dpi"], bbox_inches="tight")
    plt.close()

    # ── 4c. Heatmap tương quan toàn bộ metrics
    corr_map = {
        "V1 Orig Mean"    : "v1_orig_mean",    "V1 CLAHE Mean"   : "v1_clahe_mean",
        "V1 Orig Std"     : "v1_orig_std",     "V1 CLAHE Std"    : "v1_clahe_std",
        "V1 CLAHE Entropy": "v1_clahe_entropy",
        "V2 CLAHE Mean"   : "v2_clahe_mean",   "V2 CLAHE Std"    : "v2_clahe_std",
        "V2 CLAHE Entropy": "v2_clahe_entropy",
        "Δ Mean V1"       : "v1_delta_mean",   "Δ Entropy V1"    : "v1_delta_entropy",
    }
    avail = {k: v for k, v in corr_map.items() if v in stats_df.columns}
    corr_data = stats_df[list(avail.values())].rename(
        columns={v: k for k, v in avail.items()})

    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(corr_data.corr(), ax=ax, annot=True, fmt=".2f", cmap="coolwarm",
                center=0, square=True, linewidths=0.5, annot_kws={"size": 8})
    ax.set_title("Heatmap Tương quan Metrics (cả 2 View + Δ)", fontweight="bold")
    plt.tight_layout()
    plt.savefig(out / "4_split_analysis" / "correlation_heatmap.png",
                dpi=cfg["dpi"], bbox_inches="tight")
    plt.close()

    # ── 4d. Summary table
    summary_rows = []
    for split in splits:
        sub = stats_df[stats_df["split"] == split]
        if sub.empty:
            continue
        row = {"split": split, "n_samples": len(sub)}
        for v in [1, 2]:
            for ph in ["orig", "clahe"]:
                for m in ["mean", "std", "entropy"]:
                    col = f"v{v}_{ph}_{m}"
                    if col in sub.columns:
                        row[col] = round(sub[col].mean(), 4)
            for m in ["mean", "std", "entropy"]:
                col = f"v{v}_delta_{m}"
                if col in sub.columns:
                    row[col] = round(sub[col].mean(), 4)
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows).set_index("split")
    summary_df.to_csv(out / "4_split_analysis" / "split_summary.csv")
    print("\n" + "=" * 70)
    print(summary_df.T.to_string())
    print("=" * 70)
    print(f"   ✔ split_summary.csv → {out / '4_split_analysis'}")


# ─────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────

def main():
    print("=" * 65)
    print("  CLAHE Medical Imaging — EDA Pipeline")
    print(f"  Clip={CONFIG['clahe_clip']}  |  Tile={CONFIG['clahe_tile']}")
    print("=" * 65)

    out = setup_dirs(CONFIG["output_dir"])
    df  = load_annotations(CONFIG["annotation_path"])

    if df.empty:
        print("❌ Không có dữ liệu. Kiểm tra lại annotation_path.")
        return

    plot_before_after(df, out, CONFIG)              # 1
    plot_histograms(df, out, CONFIG)                # 2
    stats_df = compute_stats(df, out, CONFIG)       # 3
    plot_split_analysis(df, stats_df, out, CONFIG)  # 4

    print(f"""
✅ Hoàn tất! Kết quả tại: ./{CONFIG['output_dir']}/

    eda_output/
    ├── 1_before_after/
    │   ├── before_after_train.png  ← [V1 gốc|V1 CLAHE|V2 gốc|V2 CLAHE|Diff]
    │   ├── before_after_val.png
    │   └── before_after_test.png
    ├── 2_histograms/
    │   ├── pdf_overlay_all_splits.png   ← PDF tất cả split, 2 view
    │   ├── pdf_cdf_train.png
    │   ├── pdf_cdf_val.png
    │   └── pdf_cdf_test.png
    ├── 3_statistics/
    │   ├── pixel_stats.csv              ← mọi metrics từng mẫu
    │   ├── boxplot_mean/std/entropy.png ← phân phối theo split
    │   └── delta_kde.png               ← Δ KDE: 2 view × 3 metrics
    └── 4_split_analysis/
        ├── split_distribution.png       ← bar + pie chart
        ├── scatter_view1_vs_view2.png   ← tương quan 2 góc chụp
        ├── correlation_heatmap.png      ← heatmap metrics
        └── split_summary.csv           ← bảng tổng hợp
""")


if __name__ == "__main__":
    main()

  CLAHE Medical Imaging — EDA Pipeline
  Clip=2.0  |  Tile=(8, 8)
✅ Đã load 2955 mẫu
   • train : 2069 mẫu
   • val   : 296 mẫu
   • test  : 590 mẫu

📸 [1/4] So sánh Trước/Sau CLAHE...
   ✔ train → eda_output\1_before_after\before_after_train.png
   ✔ val   → eda_output\1_before_after\before_after_val.png
   ✔ test  → eda_output\1_before_after\before_after_test.png

📊 [2/4] Phân tích Histogram...
   ✔ train → PDF + CDF saved
   ✔ val   → PDF + CDF saved
   ✔ test  → PDF + CDF saved

📐 [3/4] Tính thống kê pixel...


   Stats: 100%|██████████| 2955/2955 [1:13:22<00:00,  1.49s/it]


   ✔ CSV → eda_output\3_statistics\pixel_stats.csv
   ✔ Boxplots + Delta KDE saved

📈 [4/4] Phân tích theo Split...

split                 train       val      test
n_samples         2069.0000  296.0000  590.0000
v1_orig_mean       129.2727  130.1874  128.5206
v1_orig_std         55.3329   54.2442   54.4104
v1_orig_entropy      7.3095    7.2760    7.3027
v1_clahe_mean      129.7645  130.6029  128.9983
v1_clahe_std        58.1181   57.4908   57.4846
v1_clahe_entropy     7.4295    7.4144    7.4321
v1_delta_mean        0.4918    0.4156    0.4778
v1_delta_std         2.7853    3.2467    3.0742
v1_delta_entropy     0.1200    0.1384    0.1295
v2_orig_mean       116.7584  116.4717  116.4500
v2_orig_std         65.9331   65.7077   65.6774
v2_orig_entropy      6.9988    6.9866    6.9804
v2_clahe_mean      118.7375  118.6265  118.4149
v2_clahe_std        65.8480   65.6478   65.6961
v2_clahe_entropy     7.1534    7.1497    7.1488
v2_delta_mean        1.9791    2.1548    1.9650
v2_delta_std       

In [8]:
import json

with open(CONFIG["annotation_path"], "r", encoding="utf-8") as f:
    data = json.load(f)

# Kiểm tra kiểu dữ liệu gốc
print("Type:", type(data))

# Nếu là dict → xem các key
if isinstance(data, dict):
    print("Keys:", list(data.keys()))
    # Xem thử value đầu tiên
    first_key = list(data.keys())[0]
    print(f"\ndata['{first_key}'] type:", type(data[first_key]))
    print(f"data['{first_key}'] sample:", str(data[first_key])[:300])

# Nếu là list → xem phần tử đầu
elif isinstance(data, list):
    print("Length:", len(data))
    print("First item type:", type(data[0]))
    print("First item:", str(data[0])[:300])

Type: <class 'dict'>
Keys: ['train', 'val', 'test']

data['train'] type: <class 'list'>
data['train'] sample: [{'id': 'CXR2384_IM-0942', 'report': 'The heart size and pulmonary vascularity appear within normal limits. A large hiatal hernia is noted. The lungs are free of focal airspace disease. No pneumothorax or pleural effusion is seen. Degenerative changes are present in the spine.', 'image_path': ['CXR2
